In [1]:
import requests
import json
import csv
import time
from typing import Optional

paper_pmid = "38790019"
paper_doi  = "10.1186/s13023-024-03213-x"
paper_pmc  = "PMC11127317"

seed_genes = [
    {"symbol": "RRAGD",  "hgnc_id": "HGNC:19903"},
    {"symbol": "COL4A3", "hgnc_id": "HGNC:2204"},
    {"symbol": "NPHS2",  "hgnc_id": "HGNC:13394"},
    {"symbol": "HNF1A",  "hgnc_id": "HGNC:11621"},
    {"symbol": "APOL1",  "hgnc_id": "HGNC:618"},
]

paper_disease_map = {
    "RRAGD":  "Hypomagnesemia, tubulopathy, and dilated cardiomyopathy (electrolyte-losing tubulopathy; mTOR signaling activation)",
    "COL4A3": "Alport syndrome, recessive and dominant forms (MIM 203780 and MIM 104200)",
    "NPHS2":  "Autosomal recessive nephrotic syndrome type 2 (MIM 600995)",
    "HNF1A":  "Likely pathogenic variant possibly explaining patient's partial phenotype (specific disease not named in paper)",
    "APOL1":  "Kidney disease risk in individuals of African ancestry (G1/G2 polymorphic risk alleles)",
}

hgnc_base    = "https://rest.genenames.org"
ensembl38    = "https://rest.ensembl.org"
ensembl_hg19 = "https://grch37.rest.ensembl.org"

def hgnc_fetch(hgnc_id: str) -> dict:
    """Fetch gene record from the HGNC REST API by HGNC accession ID."""
    numeric_id = hgnc_id.replace("HGNC:", "")
    url = f"{hgnc_base}/fetch/hgnc_id/{numeric_id}"
    resp = requests.get(url, headers={"Accept": "application/json"}, timeout=15)
    resp.raise_for_status()
    docs = resp.json().get("response", {}).get("docs", [])
    return docs[0] if docs else {}


def ensembl_coords(symbol: str, server: str) -> Optional[dict]:
    """Return genomic coordinates from Ensembl REST lookup."""
    url = f"{server}/lookup/symbol/homo_sapiens/{symbol}"
    try:
        resp = requests.get(url, headers={"Content-Type": "application/json"}, timeout=15)
        if resp.status_code == 200:
            return resp.json()
    except requests.RequestException:
        pass
    return None

def format_coords(data: Optional[dict]) -> str:
    if not data:
        return "N/A"
    chrom  = data.get("seq_region_name", "?")
    start  = data.get("start", "?")
    end    = data.get("end", "?")
    strand = "+" if data.get("strand", 1) == 1 else "-"
    return f"chr{chrom}:{start}-{end} ({strand})"

def process_genes(genes: list) -> list:
    results = []
    for gene in genes:
        symbol  = gene["symbol"]
        hgnc_id = gene["hgnc_id"]

        hgnc_record = hgnc_fetch(hgnc_id)
        time.sleep(0.4)

        hgnc_name = hgnc_record.get("name", "N/A")
        alias_parts = (
            hgnc_record.get("prev_symbol", [])
            + hgnc_record.get("alias_symbol", [])
        )
        aliases = "; ".join(alias_parts) if alias_parts else "N/A"

        hg38_coords = format_coords(ensembl_coords(symbol, ensembl38))
        time.sleep(0.4)

        hg19_coords = format_coords(ensembl_coords(symbol, ensembl_hg19))
        time.sleep(0.4)
        
        disease = paper_disease_map.get(symbol, "Not mentioned in paper")

        results.append({
            "hgnc_id":        hgnc_id,
            "hgnc_gene_name": hgnc_name,
            "gene_symbol":    symbol,
            "gene_aliases":   aliases,
            "hg38_coords":    hg38_coords,
            "hg19_coords":    hg19_coords,
            "disease":        disease,
        })
    return results

def write_json(results: list, path: str) -> None:
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(
            {
                "pmid":  paper_pmid,
                "doi":   paper_doi,
                "pmc":   paper_pmc,
                "genes": results,
            },
            fh, indent=2,
        )

In [2]:
results = process_genes(seed_genes)
write_json(results, "gene_output.json")
for r in results:
    print(f"  {r['hgnc_id']:12s} | {r['gene_symbol']:8s} | hg38: {r['hg38_coords']}")
print("Output: gene_output.json")

  HGNC:19903   | RRAGD    | hg38: chr6:89364616-89412735 (-)
  HGNC:2204    | COL4A3   | hg38: chr2:227164624-227314792 (+)
  HGNC:13394   | NPHS2    | hg38: chr1:179550494-179575952 (-)
  HGNC:11621   | HNF1A    | hg38: chr12:120978543-121002512 (+)
  HGNC:618     | APOL1    | hg38: chr22:36253071-36267530 (+)
Output: gene_output.json
